<a href="https://colab.research.google.com/github/DayDreamChaser/llm-code/blob/main/Agent/Compress/LLMLingua2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LLMLingua2

<a target="_blank" href="https://colab.research.google.com/github/microsoft/LLMLingua/blob/main/examples/LLMLingua2.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<a target="_blank" href="https://aclanthology.org/2024.findings-acl.57/">LLMLingua-2</a> focuses on task-agnostic prompt compression for better generalizability and efficiency. It is a small-size yet powerful prompt compression method trained via data distillation from GPT-4 for token classification with a BERT-level encoder, excels in <b>task-agnostic compression</b>. It surpasses LLMLingua in handling <b>out-of-domain data</b>, offering <b>3x-6x faster</b> performance.

Below, We showcase the usage and compression results of <i>LLMLingua-2</i> on both <b>in-domain</b> and <b>out-of-domain</b> datasets, including various tasks such as single-document QA, multi-document QA, summarization and in-context learning.


In [1]:
!pip install llmlingua

from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
    use_llmlingua2=True,
)

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Target LLM Config

In [2]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.45.0
    Uninstalling openai-2.45.0:
      Successfully uninstalled openai-2.45.0


In [ ]:
# Using the OAI
import openai

openai.api_key = "<insert_openai_key>"

# or Using the AOAI
import openai

openai.api_key = "<insert_openai_key>"
openai.api_base = "<insert_openai_base>"
openai.api_type = "azure"
openai.api_version = "2023-05-15"

## In-Domain

Below, we present the results of <i>LLMLingua-2</i> compared to the strong baselines on In-Domain data: test set of <a href="https://aclanthology.org/2023.acl-long.906/">MeetingBank</a>.
Despite the fact that our compressors are much smaller than the LLaMa-2-7B used in the baselines,
our approach achieves <b>significantly better performance</b> on both the QA and Summary tasks, and <b>comes close to matching the performance of the original prompt</b>.

### MeetingBank


In [3]:
# Download the original prompt and dataset
from datasets import load_dataset

dataset = load_dataset("huuuyeah/meetingbank", split="test")
context = dataset[0]["transcript"]

question = "What is the agenda item three resolution 31669 about?\nAnswer:"
reference = "Encouraging individualized tenant assessment."

README.md:   0%|          | 0.00/3.32k [00:00<?, ?B/s]

train.json: reconstructing file:   0%|          |  0.00B / 88.4MB            

train.json: downloading bytes:           |  0.00B            

validation.json: reconstructing file:   0%|          |  0.00B / 13.2MB            

validation.json: downloading bytes:           |  0.00B            

test.json: reconstructing file:   0%|          |  0.00B / 13.4MB            

test.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/861 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/862 [00:00<?, ? examples/s]

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt = "\n\n".join([context, question])

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [2]:
# context = '''
# 下面的回答分三块：

# 1. 先把你说的四个“流派”系统化补完，并结合近期论文/产品评价一下优劣和适用场景
# 2. 结合银行数字员工（多轮工具调用、内网 DeepSeek‑V4‑Flash）分析哪几类技术在工程上真有性价比
# 3. 给出一个「可落地的」上下文优化 / 压缩方案（含策略、数据结构、触发机制、以及如何避坑）

# ---

# ## 一、四类上下文 / Prompt 压缩流派系统梳理与评价

# ### 1. 直接调大模型做提示词压缩 / SFT/DPO/RLHF 学会压缩

# **典型做法**

# - 让同一个或更强的 LLM执行：
#   - 单轮/多轮 prompt → “请在不损失关键信息的前提下，把以上内容压缩到 N tokens 内”
#   - 进一步在系统提示里约束结构（如 bullet points、JSON schema、专用 DSL）
# - 再往前一步，是专门训练一个「压缩专家」：
#   - 用 GPT‑4 等老师模型标注大量 `(原上下文, 任务指令) → 最佳压缩上下文`，对小模型做 SFT 或 DPO
#   - 也有工作把「是否保留某段上下文」变成二分类，让模型学会「压缩决策策略」

# **优点**

# - 极其灵活：可以整段总结、按业务字段抽取、保留计算中间推理链等
# - 训练/推理管线比较简单：就是「先调一次压缩 LLM，再调任务 LLM」的两阶段调用
# - 非常适合**高度结构化的业务压缩模板**：
#   - 例如银行对话里，永远关注：客户身份 → 业务目标 → 关键数值 → 风险点 → 工具执行结论
#   - 可以把这些字段变成 schema，强制压缩输出成结构化结果，再拼接到下游 prompt

# **缺点**

# - 成本：每次压缩本身又是一次 LLM 调用，对你现在 10w token 的场景，如果还是用 DeepSeek‑V4‑Flash 做压缩，就会非常慢
# - 延迟：多轮 Agent 风格对话里，如果「每一轮都全量 summarization」，会直接毁掉用户体验
# - 信息丢失不可控：如果完全交由大模型自由“摘要”，在金融场景非常危险——数字、阈值、日期经常被「合理化」或忽略。近期有专门论文在金融分析场景上验证：LLM 摘要会产生“看起来很合理但改变决策”的压缩结果[1]，对风控与合规来说是红线。

# **什么时候可行**

# - **压缩频率低 + 有硬结构模版**时：例如每 8～10 轮对话，对「历史对话 + 工具轨迹」做一次**结构化总结**；
# - 但压缩模型最好不要用你同一台 DeepSeek‑V4‑Flash，而是一个更小的 encoder/指令微调模型（甚至 1–3B 级别在 CPU/GPU 上跑），否则你现在遇到的延迟问题只会更严重。

# ---

# ### 2. 硬压缩：LLMLingua 系列、PPL/打分 + Token Pruning

# **典型代表**

# - LLMLingua 1：用一个小的 LM 计算每个 token 的困惑度/信息熵，根据阈值裁剪低信息 token[2]
# - LLMLingua‑2：不再直接用 PPL，而是把每个 token 的“保留 / 丢弃”变成双向 encoder（如 XLM‑R / BERT）的分类任务，用 GPT‑4 蒸馏得到训练数据，并支持 query‑aware、文档级压缩[3]
# - 其他同类工作：Selective-Context、基于 Diffusion 的 token-level prompt pruning 等

# **优点**

# - 工程链路成熟，有现成开源实现，2026 年业界仍普遍把 LLMLingua 当作 **“Prompt 压缩基础线”**[3]
# - 推理阶段可以完全在小模型上做，理论上非常适合你说的「资源有限、LLM 在内网」的情况
# - 对**自然语言长文档 RAG 提示**非常有效，20× 压缩下仍能保留多数下游性能[3]

# **短板 / 在你场景里为什么「不咋行」**

# - 这类方法更适合**非结构化文字**（百科、网页、长报告），而不是：
#   - 工具 JSON 返回（字段名很多，但都很重要）
#   - 代码 / 日志（任何一行删错就会导致不可恢复的语义错误）
# - token 级剪枝很难表达**流程状态**：
#   在 Agent 里，你真正关心的是“第几轮调用了哪个工具 + 结果是什么 + 之后如何被利用”，这更像**轨迹级/回合级信息**，而不是某些词重要、某些词不重要。
# - 从最新综述来看，硬剪枝常常在 4–6× 以上压缩就出现「明显语义崩塌」，尤其对对齐过的指令模型更明显[4]。

# **结论**

# - 在你的银行数字员工场景，**不建议把 LLMLingua 这类硬压缩直接应用于工具结果 / JSON / 代码本身**
# - 更合适的用法是：对**外部检索来的长文本资料**做预压缩：政策条文、FAQ、市场分析报告等，辅助数字员工引用，但这跟你现在的「多轮工具调用轨迹」问题是两个层面。

# ---

# ### 3. 语义软压缩：DeepSeek‑OCR、C3（Context Cascade Compression）、LCLM 等

# 你提到得很对，这一类其实是在做「**编码器‑解码器式的语义瓶颈**」——把长上下文压成一小串**潜在向量 / latent token**，再喂给大模型。

# **代表工作**

# - DeepSeek‑OCR：把文本渲染成图像，再由视觉 encoder 编成几十到几百个 vision token 送入语言 decoder，实现 10× 以上压缩[5]；本质上是「视觉化上下文压缩」，在长 PDF / 文档场景对 token 成本非常友好
# - DeepSeek‑OCR‑2 则进一步让 visual encoder 学会“动态 token 重排”，压缩率更高[6]
# - C3（Context Cascade Compression）：美团系工作，cascade 两个 LLM，小 LLM 作为 encoder，把长文本 → textual latent tokens，再由大 LLM 作为 decoder完成下游任务，可做到 20× 甚至 40× 压缩，仍保留 60%+ 性能[7]
# - 更泛化的一类是 LCLM（Latent Context Language Models），统一 encoder‑decoder 框架，把整个长序列编码成一个压缩 latent，再解码[8]

# **优点**

# - 这是当前学术界证明「高倍率压缩仍可工作」的主力路线：
#   C3 在长文本 QA 上 20× 压缩还能保持较高准确率[7]，LCLM 在 16× 压缩下仍优于多种 KV cache 方案[8]
# - encoder 可以是**小模型 + 单次前向**，有机会在你内网环境里单独部署一个小 encoder，而把昂贵的 DeepSeek‑V4‑Flash留给真正回答阶段
# - 天然适合做**语义级别的软压缩**，比 token 剪枝更易保留语义一致性

# **银行 Agent 场景下的问题**

# - 当前公开 C3 / LCLM 模型主要围绕**文本 QA / 一般推理**评测，对「多轮工具调用轨迹、JSON、代码、SQL」这类高度结构化交互的鲁棒性尚不充分
# - C3 这类方法在工程上意味着：你要在内网跑两个 LLM（encoder LLM + decoder LLM）。对你现在说的「资源有限」场景，**比起再加一个 LLM，反而不如加一个小 encoder（50M–1B）+ 轻量规则来得现实**
# - 训练数据需求大：要想让 encoder‑latent 能正确表达「金融流程状态 / 工具调用意图」，通常需要针对你们银行业务流程重新蒸馏一轮，而不是直接拿通用 C3 模型就用。

# **结论**

# - 语义软压缩的思想非常值得借鉴：**“用较小的 encoder 把长轨迹压缩成结构化 latent，再让大模型读 latent”**
# - 但在你短期工程实践中，可以做一个**「半软压缩」**：用小 encoder + 规则，从轨迹里抽出关键字段（工具名、时间、输入输出要点）形成**结构化 summary**，而不是全套 latent‑decoder 训练。

# ---

# ### 4. Agent Harness / Headroom / Context OS 一类工程化上下文管理

# 这里其实不是单一算法，而是一整套「**Agent 上下文工程框架**」。

# **关键能力（综合多篇介绍和产品文档）**

# - 对不同数据类型有专门压缩器：
#   - JSON：字段级选择 + minify + token 优化格式（比如 TOON 代替 JSON）[9]
#   - 代码：只保留当前工作文件 + 引用的少量依赖，长堆栈 /日志通过 anchored summarization压到几行要点[10]
#   - 纯文本：LLM 摘要 + 段落级选取
# - 「轨迹级」压缩：把多轮工具调用历史整合为有限长度的**执行摘要**，包括：
#   - 用过哪些工具、成功/失败、关键中间结论
#   - 用结构化 schema 存：`[ {step, tool, input_key_points, output_key_points, decision} , ... ]`
# - 自动触发：
#   - Codex / Claude Code / OpenCode 这类 harness 在接近上下文上限时自动触发 compaction，而不是每一轮都跑压缩[11]
# - Headroom 一类专门的压缩层：
#   - 作为代理层，拦截 agent 的工具输出 / RAG 结果 / 文件内容，在进入 LLM 之前先压缩，号称可减少 60–95% token[12]
#   - **可区分可逆 / 不可逆压缩**：对某些内容只放一个 placeholder（同时在本地存全量结果，必要时再 `headroom_retrieve` 取回[13]）

# **优点**

# - 和你的场景匹配度最高：他们也是围绕「coding agent / terminal agent / multi‑tool agent」做上下文治理
# - 工程实践充分：已有大量经验表明：
#   - 单靠 RAG + 长上下文，不做上下文工程，token 成本和延迟都会爆炸[14]
#   - 通过“写、选、压缩、隔离”的组合，能把多轮会话的 tokens 控制在一个稳定区间

# **缺点**

# - 本质需要你在自己系统里重建一套「简化版 Headroom + Harness」：
#   - 对不同数据源写压缩策略
#   - 设计轨迹摘要结构 & 触发阈值
#   - 对压缩失真要做专门的评估（特别是合规/风控任务）

# **对你来说的最大价值**

# - 这类框架已经给出了「**Agent 上下文管理的分层思路**」：
#   - Working Memory（当前几轮 + 关键工具结果）
#   - Compressed History（多个阶段性 summary）
#   - External Memory（数据库 / RAG / 文档等）
# - 只要把你现在「分块压缩 + 再评价」的做法重构到这个分层框架里，就能显著减少你用 V4‑Flash 做压缩的次数和长度。

# ---

# ## 二、结合你的约束条件，哪些技术是真正可行的？

# 你的约束和现状：

# - 场景：**银行数字员工 Agent**，有明显的合规/审计要求
# - 交互：**多轮工具调用**（查账户、查交易、算利率、风控决策、生成说明等）
# - 模型：内网部署 **DeepSeek‑V4‑Flash**（284B MoE，长上下文），但硬件/并发有限
# - 问题：10w token 做压缩太慢，现在做法是「把长上下文分块，再用 V4‑Flash 自己压缩和评价」——相当于用重炮打蚊子

# 在这个前提下：

# ### 不推荐作为主力的

# 1. **让 V4‑Flash 自己做大段 summarization / 压缩**
#    - 成本高、延迟大，还占用同一集群资源；在高并发生产银行场景里不可持续
# 2. **对工具 JSON / 代码 / 日志直接用 LLMLingua 式 token 裁剪**
#    - 风险太大，很容易断掉字段间依赖，破坏审计可追溯性
# 3. **短期内直接上 C3 / LCLM 全套 encoder‑decoder 压缩链**
#    - 需要双模型 + 大量业务蒸馏训练，目前公开模型在金融 multi‑tool Agent 场景没有成熟 best practice

# ### 推荐主轴：**Agent Harness + 轻量语义压缩 + 结构化摘要**

# 核心思路：

# - 把「长上下文问题」拆成三类对象处理：
#   1. **对话轮次**（customer <-> agent）
#   2. **工具调用轨迹**（Tool calls + Tool results）
#   3. **外部长文档 / RAG 结果 / 历史报告**
# - 在每一类对象上，用**不同层级的压缩策略**，并尽量用**小模型 + 规则**代替大模型做压缩。

# ---

# ## 三、面向银行数字员工的上下文优化方案（建议架构）

# 下面给一个可以工程落地的方案，你可以理解为「在你自家系统里，用 Headroom/Agent Harness 思路搭一套简化版上下文层」，尽量避免 10w token 级的 V4‑Flash 压缩调用。

# ### 1. 上下文整体分层设计

# 建议把 Agent 的「记忆 / 上下文」分成四层：

# 1. **即时上下文（Working Set）**
#    - 最近 N 轮对话（例如 4～6 轮）
#    - 最近 K 次「关键工具调用 + 结果」的**完整内容**（例如 3～5 条）
# 2. **阶段性执行摘要（Trajectory Summaries）**
#    - 每当工具调用轨迹超过一定长度（比如 8～10 次），就把前面的调用压缩成一个 structured summary，替换掉原始结果
# 3. **外部长文档 / RAG 结果**
#    - 永远按需检索 + 预压缩，禁止“整篇 doc 永久呆在上下文里”
# 4. **持久业务内存 / 审计存档（Off‑Context Memory）**
#    - 全量对话日志、工具输入输出、模型回复，都写入审计库
#    - 上下文里只放标识符（如 `HISTORY_ID=xxx`），需要时通过专门工具（如 `retrieve_audit_context`）取回

# **关键原则**：
# > LLM 的上下文是「工作记忆」，而不是「永久记忆」。所有长久存储、审计要求，都丢到外部系统（DB / 日志）。

# ---

# ### 2. 针对多轮工具调用结果的专用压缩策略

# #### 2.1 工具调用数据结构

# 建议为每次工具调用维护一个统一 schema（存数据库 + 存在上下文里可压缩表示）：

# ```json
# {
#   "step_id": "T12",
#   "timestamp": "...",
#   "tool_name": "query_transactions",
#   "input_summary": "客户 12345，近 90 天交易，额度 > 10,000",
#   "output_schema_version": "v1",
#   "output_key_fields": {
#     "txn_count": 4,
#     "total_amount": 520000.00,
#     "risk_flags": ["large_cash", "international_wire"],
#     "samples": [
#       {"date": "2026-06-01", "amount": 200000, "country": "US"},
#       {"date": "2026-07-15", "amount": 320000, "country": "CN"}
#     ]
#   },
#   "raw_output_ref": "AUDIT_DB:transactions:session123:T12",
#   "used_by_steps": ["T15", "T18"]
# }
# ```

# - `raw_output_ref`：指向审计库中完整 JSON / 报文
# - 上下文中只保留 `output_key_fields`（少量字段 + 样例），如果后续模型**确实需要完整数据**，可以通过一个工具 `fetch_raw_tool_result(ref, filter)` 去 DB 再拉。

# #### 2.2 压缩策略（分层）

# 1. **新工具结果刚产生时**
#    - 不做压缩，完整“关键字段版本 + 样例”入上下文
# 2. **当上下文长度接近阈值（例如 60k tokens）时触发阶段性 compaction**
#    - 对「最早的若干工具调用」执行：
#      - 保留：`tool_name`, `input_summary`, `output_key_fields` 的小字典（控制在几十个 token）
#      - 删除：样例列表、冗长说明、错误回溯
#      - 确保仍然能回答：**“之前是否查过 X？结果大致是什么？”**
# 3. **对纯日志 / 非关键内容**
#    - 直接移出上下文，只在审计库中保留，并留下如：`[LOGS_REMOVED, REF=...]`

# 这样，整个工具轨迹就从「线性堆积」变成了「关键元信息 + 审计引用」，大幅降低了要送进 DeepSeek‑V4‑Flash 的 token 数。

# #### 2.3 小模型辅助的语义软压缩（可选）

# - 可以考虑在内网部署一个 **小 encoder（0.5–3B）**，专门对「工具输入/输出自然语言部分」做 embedding，再只存一小段 top‑K 代表句子
# - 或者把多次类似查询（同一客户不同时间）合并为「聚合摘要 + 最近几次明细」

# 这相当于在 Agent Harness 思想之上，加了一点「C3 / LCLM 式语义瓶颈」，但只对**工具的描述部分**，不碰关键字段和金额。

# ---

# ### 3. 对话 / 多轮交流的上下文策略

# #### 3.1 滑动窗口 + anchored summarization

# - 始终保留：
#   - 最近 4–6 轮用户与 Agent 的**原文对话**
#   - 初次系统提示 / 角色设定（含合规提示）
# - 对更早的对话，做成一个或若干个「阶段性摘要」，每个摘要包括：
#   - 用户目标演化（如“从查询余额 → 咨询贷款 → 申请产品 X”）
#   - 已确认的关键信息（如身份认证结果、收入水平、已有产品）
#   - 已经明确的合规约束（禁止建议、风险披露等）

# 可以由小模型或 V4‑Flash 在**低频触发**下生成这类摘要，例如：

# - 规则：每超过 10 轮对话，或者执行完一个“业务任务”（如一次贷款审批），生成一条新的 summary 替换旧对话内容。

# #### 3.2 与工具轨迹的联动

# - 在生成对话摘要时，不要重新读 10w token 全历史
#   - 而是只读「阶段性执行摘要（trajectory summaries）」+ 最近几轮对话
# - 这保证了任何一次 summarization 调用的输入不超过几千 token，避免你现在 10w token 压缩的灾难。

# ---

# ### 4. 外部文档 / RAG 的压缩

# 虽然你问题主要在工具调用，但银行数字员工不可避免会读大量文档（政策、条款、FAQ）。这里可以充分利用硬压缩 / 语义压缩：

# 1. **把文档预处理到向量库时就做 LLMLingua / 语义摘要**
#    - 每个 chunk 压到 1/4 tokens 甚至更少
#    - 同时保留一段「结构化元数据」（条款编号、适用人群、金额上限等）
# 2. **在召回阶段，只把少量最相关 chunks + 元数据送进上下文**
#    - 绝不把原始 20 页条款一次性塞进 DeepSeek 的 context

# 这部分和你的工具轨迹上下文是解耦的，只要 RAG 层自己做好压缩，就不会挤占 Agent 工具记忆预算。

# ---

# ### 5. 压缩触发与阈值设计

# 参考 ACON 和 Agent Harness 经验[15][16]，建议不要「每一轮都压」，而是：

# - 设置三个阈值：
#   1. **软阈值**（如 50k tokens）：只做轻量操作（清理旧日志、去掉明显无用的工具调用结果）
#   2. **硬阈值**（如 80k tokens）：触发阶段性 compaction：
#      - 对最旧一段对话 + 工具轨迹生成 structured summary
#      - 移除原始内容
#   3. **紧急阈值**（如 90k tokens）：拒绝再添加非必要内容，如直接在系统层拒绝超长 RAG 注入

# 触发压缩时的执行顺序建议：

# 1. 先清理 / minify JSON（删除空字段、多余空格、长字段名可做别名映射），这一类完全**无需 LLM**
# 2. 再对日志 / 非关键工具输出做裁剪（只留 error code + 1 行摘要）
# 3. 最后才动 LLM summarization（对对话与轨迹）

# ---

# ### 6. 模型选择与部署建议

# 结合你「ds‑v4‑flash 资源有限」这一点：

# 1. **DeepSeek‑V4‑Flash**
#    - 只用于真正的「业务回答 / 推理」
#    - 不参与绝大多数压缩流程
# 2. **小模型（本地 encoder / instruction model）**
#    - 用于：
#      - 工具输入/输出的自然语言简要化
#      - 对话阶段性摘要
#      - 关键词提取 / slot filling
#    - 模型规模可以是 1–3B，单机 CPU 也可以撑得住
# 3. **规则 + 程序化压缩**
#    - JSON minify / TOON 化 / 字段选取
#    - 日志裁剪、代码摘取
#    - 这些全部用程序完成，不用任何 LLM 资源

# 如果暂时没有合适小模型，也可以先只做**规则 + 结构化压缩**，已经能把大量无关 token（JSON 括号/引号、重复日志）去掉，往往有 40–60% 的节省空间[9][12]。

# ---

# ### 7. 评估与合规保障

# 在银行场景，压缩方案需要通过**定量 + 定性**评估：

# 1. **定量指标**
#    - 平均每任务 token 数（包括压缩 + 推理）
#    - 成功率 / 准确率：在典型业务流程（开户、转账风控、贷款审批）上的任务完成情况
#    - 再现性：把压缩前 / 后的上下文都送给 LLM，看结论差异
# 2. **定性 & 合规**
#    - 针对金融决策场景（授信额度、拒贷原因、风险评级），人工比对压缩前后的结论
#    - 重点看：金额、日期、阈值是否被篡改或丢失
#    - 对有所怀疑的压缩策略（比如对数字密集的表格摘要），要么禁止，要么保留低压缩率版本。

# 从近期研究看，金融领域对 LLM 压缩后的信息保真性非常敏感[1]，监管与内部模型风险管理都要求「任何用于决策的自动处理不能隐藏或改变关键事实」，因此：

# > 对涉及金额 / 风控的工具结果，**只允许字段级压缩（去掉没用字段），不允许语义重写**。
# > 可以对解释性话术用 LLM 摘要，但原始数字要么保留在上下文，要么可通过引用随时查询。

# ---

# ## 四、小结：给你一个落地版本的「路线图」

# 如果要给一个 3 个月内可落地的目标，我会建议：

# 1. **第 0 阶段：立刻做的工程优化**
#    - 引入统一的 `ToolCallRecord` 结构 + 审计引用
#    - 对 JSON / 日志 / 代码做程序化裁剪和 minify
#    - 实现基于 token 估算器的软/硬阈值触发逻辑
# 2. **第 1 阶段：结构化压缩上线**
#    - 实现「阶段性 execution summary」：基于规则从多个 ToolCallRecord 提取：工具序列、关键数字、风险标记
#    - 把你现在 10w token V4‑Flash 压缩流程下线，改成：每次 summary 输入不超 2k–4k token
# 3. **第 2 阶段：引入小模型做轻量语义压缩**
#    - 部署一个小 encoder / 指令模型，专门负责对话摘要和工具描述提炼
#    - 对长文档/RAG 层引入 LLMLingua‑style 或其他 prompt 压缩
# 4. **第 3 阶段：评估与调优**
#    - 针对若干关键业务使用 FinanceBench / FinToolBench 等金融任务或自建测试集验证压缩前后性能差异[17][18]
#    - 对「压缩导致决策偏移」的情形做专门记录和规则修正

# 这样走下来，你实际上是采纳了：

# - 第 1 流派的一部分（结构化摘要 + 小模型压缩）
# - 第 2/3 流派的思想（只在合适的对象上做语义软压缩，而不是 token 剪枝）
# - 第 4 流派的工程实践（Agent Harness + Headroom 样式的上下文层）

# 同时又避免了：在 DeepSeek‑V4‑Flash 上做任何超长压缩推理，从根本上解决你现在的「10w token 压缩太慢」问题。
# '''
context = '''
User: 帮我查一下上周（5月20号到26号）的DAU、新注册用户数，还有每天的付费转化率，看看哪天转化率最高。
Assistant: 好的，我将从 dwd_user_active_log（用户活跃日志）、dim_user_info（用户画像表）和 dwd_pay_order（支付订单表）中查询相关数据。现在执行 SQL 查询。
{
    "tool_name": "execute_sql",
    "arguments": {
        "sql": """
        WITH daily_metrics AS (
            SELECT
                DATE(a.active_time) AS dt,
                COUNT(DISTINCT a.user_id) AS dau,
                COUNT(DISTINCT CASE WHEN u.reg_date = DATE(a.active_time) THEN u.user_id END) AS new_users,
                COUNT(DISTINCT o.user_id) AS paying_users
            FROM dwd_user_active_log a
            LEFT JOIN dim_user_info u ON a.user_id = u.user_id
            LEFT JOIN dwd_pay_order o ON a.user_id = o.user_id
                AND DATE(o.pay_time) = DATE(a.active_time)
                AND o.status = 'success'
            WHERE DATE(a.active_time) BETWEEN '2024-05-20' AND '2024-05-26'
            GROUP BY DATE(a.active_time)
        )
        SELECT
            dt,
            dau,
            new_users,
            paying_users,
            ROUND(paying_users * 1.0 / dau, 4) AS pay_conversion_rate
        FROM daily_metrics
        ORDER BY dt;
        """
    }
}


Tool Response (execute_sql):
dt dau new_users paying_users pay_conversion_rate

2024-05-20 152300 2100 12800 0.0840

2024-05-21 148500 1980 12100 0.0815

2024-05-22 146800 2050 11900 0.0811

2024-05-23 149200 2200 12500 0.0838

2024-05-24 168900 3500 15200 0.0900

2024-05-25 185600 4200 17800 0.0959

2024-05-26 172400 3800 16000 0.0928

第三轮：Agent 调用 Python 进行二次计算与格式化

Assistant: 查询结果显示5月25日的转化率最高。为了更直观地展示环比变化，我使用 Python 计算环比增长率并格式化输出。
{
    "tool_name": "execute_bash_tool",
    "arguments": {
        "command": "python -c '\nimport pandas as pd\nimport sys\ndata = [\n    {\"dt\": \"2024-05-20\", \"dau\": 152300, \"new_users\": 2100, \"paying_users\": 12800, \"rate\": 0.0840},\n    {\"dt\": \"2024-05-21\", \"dau\": 148500, \"new_users\": 1980, \"paying_users\": 12100, \"rate\": 0.0815},\n    {\"dt\": \"2024-05-22\", \"dau\": 146800, \"new_users\": 2050, \"paying_users\": 11900, \"rate\": 0.0811},\n    {\"dt\": \"2024-05-23\", \"dau\": 149200, \"new_users\": 2200, \"paying_users\": 12500, \"rate\": 0.0838},\n    {\"dt\": \"2024-05-24\", \"dau\": 168900, \"new_users\": 3500, \"paying_users\": 15200, \"rate\": 0.0900},\n    {\"dt\": \"2024-05-25\", \"dau\": 185600, \"new_users\": 4200, \"paying_users\": 17800, \"rate\": 0.0959},\n    {\"dt\": \"2024-05-26\", \"dau\": 172400, \"new_users\": 3800, \"paying_users\": 16000, \"rate\": 0.0928}\n]\ndf = pd.DataFrame(data)\ndf[\"rate_mom\"] = df[\"rate\"].pct_change().fillna(0)\ndf[\"rate_display\"] = (df[\"rate\"] * 100).map(\"{:.2f}%\".format)\ndf[\"mom_display\"] = (df[\"rate_mom\"] * 100).map(\"{:+.2f}%\".format)\nprint(df[[\"dt\", \"dau\", \"new_users\", \"paying_users\", \"rate_display\", \"mom_display\"]].to_string(index=False))\n'"
    }
}


Tool Response (execute_bash_tool):
         dt     dau  new_users  paying_users rate_display mom_display
2024-05-20  152300       2100         12800        8.40%      +0.00%
2024-05-21  148500       1980         12100        8.15%      -2.98%
2024-05-22  146800       2050         11900        8.11%      -0.49%
2024-05-23  149200       2200         12500        8.38%      +3.33%
2024-05-24  168900       3500         15200        9.00%      +7.40%
2024-05-25  185600       4200         17800        9.59%      +6.56%
2024-05-26  172400       3800         16000        9.28%      -3.23%


第四轮：Agent 总结关键数值信息

Assistant: 分析完成。以下是上周的核心数据洞察：

1. 整体趋势：DAU 在周末（25、26号）达到峰值，周五（24号）开始显著拉升。
2. 关键数值：
   • 最高 DAU：2024-05-25，值为 185,600。

   • 最高新注册用户：2024-05-25，值为 4,200。

   • 最高付费转化率：2024-05-25，值为 9.59%（环比增长 6.56%）。

3. 异常点：虽然26号 DAU 依然很高，但付费转化率较25号有所回落（9.28%），建议关注周末次日留存情况。

需要我进一步分析这波流量激增的原因（例如渠道来源）吗？
'''
print(context)


场景设定

用户问题：分析最近7天（2024-05-20至2024-05-26）的每日活跃用户数（DAU）、新注册用户数及付费转化率，并找出哪一天的付费转化率最高。

多轮调用上下文模拟

第一轮：用户发起查询

User: 帮我查一下上周（5月20号到26号）的DAU、新注册用户数，还有每天的付费转化率，看看哪天转化率最高。

第二轮：Agent 生成并执行 SQL

Assistant: 好的，我将从 dwd_user_active_log（用户活跃日志）、dim_user_info（用户画像表）和 dwd_pay_order（支付订单表）中查询相关数据。现在执行 SQL 查询。
{
    "tool_name": "execute_sql",
    "arguments": {
        "sql": """
        WITH daily_metrics AS (
            SELECT 
                DATE(a.active_time) AS dt,
                COUNT(DISTINCT a.user_id) AS dau,
                COUNT(DISTINCT CASE WHEN u.reg_date = DATE(a.active_time) THEN u.user_id END) AS new_users,
                COUNT(DISTINCT o.user_id) AS paying_users
            FROM dwd_user_active_log a
            LEFT JOIN dim_user_info u ON a.user_id = u.user_id
            LEFT JOIN dwd_pay_order o ON a.user_id = o.user_id 
                AND DATE(o.pay_time) = DATE(a.active_time)
                AND o.status = 'success'
            WHERE DATE(a.active_time) BETWEEN '2024-05

In [3]:
import time

start_time = time.time()
# 执行压缩
compressed_prompt = llm_lingua.compress_prompt(
    context,
    rate=0.35,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)
# 记录并计算耗时
end_time = time.time()
duration = end_time - start_time

# 统计字符数
original_char_count = len(context)
compressed_char_count = len(compressed_prompt['compressed_prompt'])
print(f"压缩耗时: {duration:.2f} 秒")
print(f"原始 Token 数: {compressed_prompt['origin_tokens']}")
print(f"压缩后 Token 数: {compressed_prompt['compressed_tokens']}")
print(f"原始字符数: {original_char_count}")
print(f"压缩后字符数: {compressed_char_count}")
print(f"Token 压缩率: {compressed_prompt['ratio']}")
print(f"字符压缩率: {original_char_count / compressed_char_count:.2f}x")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1833 > 512). Running this sequence through the model will result in indexing errors


压缩耗时: 1.27 秒
原始 Token 数: 1860
压缩后 Token 数: 656
原始字符数: 4105
压缩后字符数: 1521
Token 压缩率: 2.8x
字符压缩率: 2.70x


In [4]:
len(context)

4105

In [9]:
context

'\n场景设定\n\n用户问题：分析最近7天（2024-05-20至2024-05-26）的每日活跃用户数（DAU）、新注册用户数及付费转化率，并找出哪一天的付费转化率最高。\n\n多轮调用上下文模拟\n\n第一轮：用户发起查询\n\nUser: 帮我查一下上周（5月20号到26号）的DAU、新注册用户数，还有每天的付费转化率，看看哪天转化率最高。\n\n第二轮：Agent 生成并执行 SQL\n\nAssistant: 好的，我将从 dwd_user_active_log（用户活跃日志）、dim_user_info（用户画像表）和 dwd_pay_order（支付订单表）中查询相关数据。现在执行 SQL 查询。\n{\n    "tool_name": "execute_sql",\n    "arguments": {\n        "sql": """\n        WITH daily_metrics AS (\n            SELECT \n                DATE(a.active_time) AS dt,\n                COUNT(DISTINCT a.user_id) AS dau,\n                COUNT(DISTINCT CASE WHEN u.reg_date = DATE(a.active_time) THEN u.user_id END) AS new_users,\n                COUNT(DISTINCT o.user_id) AS paying_users\n            FROM dwd_user_active_log a\n            LEFT JOIN dim_user_info u ON a.user_id = u.user_id\n            LEFT JOIN dwd_pay_order o ON a.user_id = o.user_id \n                AND DATE(o.pay_time) = DATE(a.active_time)\n                AND o.status = \'success\'\n            WHERE DATE

In [5]:
compressed_prompt['compressed_prompt']

"\n:分析最近7天(2024-05-20至2024-05-26)的每日活跃用户数:用户发起查询 第二轮:Agent SQL\n Assistant dwd_user_active_log_user_info dwd_pay_order SQL\n_name daily_metrics\n DATE. active_time\n(DISTINCT. user_id\n. reg_date_time user_id new_users\n._id paying_users\n dwd_user_active_log\n dim_user_info._id\n dwd_pay_order.\n DATE. pay_time._time\n. status 'success\n DATE._time BETWEEN '2024-05-20' '2024-05-26'\n GROUP BY DATE.\n SELECT new_users\n paying_users\n ROUND(paying_users. pay_conversion_rate\n daily_metrics\nORDER\n Tool Response (execute\n new_users paying_conversion_rate\n 2024-05-20 152300 2100 12800.\n 2024-05-21 148500 1980 12100.\n 2024-05-22 146800 2050 11900.\n 149200 2200 12500.\n 168900 3500 15200.\n 2024-05-25 185600 4200 17800.\n 2024-05-26 172400 3800 16000.\n Python\n Python\n_bash\n -c import\n 152300_users 2100 12800.\n 148500 1980 12100.\n 146800_users 2050 11900.\n 149200_users 2200_users 12500. 0838\n 168900_users 3500_users 15200. 0900\n185600_users 4200_users 17800. 0959\n 172400_us

In [7]:
print(compressed_prompt['compressed_prompt'])


:分析最近7天(2024-05-20至2024-05-26)的每日活跃用户数:用户发起查询 第二轮:Agent SQL
 Assistant dwd_user_active_log_user_info dwd_pay_order SQL
_name daily_metrics
 DATE. active_time
(DISTINCT. user_id
. reg_date_time user_id new_users
._id paying_users
 dwd_user_active_log
 dim_user_info._id
 dwd_pay_order.
 DATE. pay_time._time
. status 'success
 DATE._time BETWEEN '2024-05-20' '2024-05-26'
 GROUP BY DATE.
 SELECT new_users
 paying_users
 ROUND(paying_users. pay_conversion_rate
 daily_metrics
ORDER
 Tool Response (execute
 new_users paying_conversion_rate
 2024-05-20 152300 2100 12800.
 2024-05-21 148500 1980 12100.
 2024-05-22 146800 2050 11900.
 149200 2200 12500.
 168900 3500 15200.
 2024-05-25 185600 4200 17800.
 2024-05-26 172400 3800 16000.
 Python
 Python
_bash
 -c import
 152300_users 2100 12800.
 148500 1980 12100.
 146800_users 2050 11900.
 149200_users 2200_users 12500. 0838
 168900_users 3500_users 15200. 0900
185600_users 4200_users 17800. 0959
 172400_users 3800 16000. 0928
 df. DataFrame
_mom

In [6]:
len(compressed_prompt['compressed_prompt'])

1521

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt = "\n\n".join([compressed_prompt["compressed_prompt"], question])

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": 100,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    model="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [8]:
eval_instruction = """
你是一个专门负责 LLM 评估的专家。下面我将为你提供两段文本：【原始上下文】和【LLMLingua-2 压缩后的上下文】。

该上下文属于一个银行“数字员工”Agent 场景，涉及多轮工具调用、客户敏感信息查询和业务逻辑推理。请根据以下维度，详细分析压缩后的文本相比原文损失了哪些关键信息，并评估其对下游 Agent 执行任务的影响：

1. **关键实体与数值损失**：是否丢失了客户 ID、交易金额、日期、利率、风控状态码等精确信息？
2. **工具调用逻辑 (Tool-calling Trace)**：压缩后是否还能清晰分辨出「哪一轮」调用了「哪个工具」，以及该工具返回的「核心结果」是什么？
3. **语义连贯性与推理链**：压缩后的文本是否变成了“词云”？Agent 是否还能根据压缩后的内容推导出正确的业务结论？
4. **合规与风险提示**：原文中涉及的合规边界、审计记录引用 (Audit Ref) 是否被破坏？

请最后给出一个 0-100 的「保真度得分」，并说明该压缩结果是否能直接用于银行生产环境。
"""

# 准备评估用的完整 Prompt
full_eval_prompt = f"""
{eval_instruction}

### 【原始上下文】
{context}

### 【LLMLingua-2 压缩后的上下文】
{compressed_prompt['compressed_prompt']}

---
请开始你的详细评估：
"""

print("评估 Prompt 已生成。请将以下内容复制到你的目标大模型（如 DeepSeek-V4-Flash）中进行评估：")
print("="*50)
print(full_eval_prompt)

# 如果需要直接调用，可以取消下面注释（前提是已配置好 API Key）
# message = [{"role": "user", "content": full_eval_prompt}]
# response = openai.ChatCompletion.create(model="gpt-4-32k", messages=message)
# print(response.choices[0].message.content)

评估 Prompt 已生成。请将以下内容复制到你的目标大模型（如 DeepSeek-V4-Flash）中进行评估：


你是一个专门负责 LLM 评估的专家。下面我将为你提供两段文本：【原始上下文】和【LLMLingua-2 压缩后的上下文】。

该上下文属于一个银行“数字员工”Agent 场景，涉及多轮工具调用、客户敏感信息查询和业务逻辑推理。请根据以下维度，详细分析压缩后的文本相比原文损失了哪些关键信息，并评估其对下游 Agent 执行任务的影响：

1. **关键实体与数值损失**：是否丢失了客户 ID、交易金额、日期、利率、风控状态码等精确信息？
2. **工具调用逻辑 (Tool-calling Trace)**：压缩后是否还能清晰分辨出「哪一轮」调用了「哪个工具」，以及该工具返回的「核心结果」是什么？
3. **语义连贯性与推理链**：压缩后的文本是否变成了“词云”？Agent 是否还能根据压缩后的内容推导出正确的业务结论？
4. **合规与风险提示**：原文中涉及的合规边界、审计记录引用 (Audit Ref) 是否被破坏？

请最后给出一个 0-100 的「保真度得分」，并说明该压缩结果是否能直接用于银行生产环境。


### 【原始上下文】

场景设定

用户问题：分析最近7天（2024-05-20至2024-05-26）的每日活跃用户数（DAU）、新注册用户数及付费转化率，并找出哪一天的付费转化率最高。

多轮调用上下文模拟

第一轮：用户发起查询

User: 帮我查一下上周（5月20号到26号）的DAU、新注册用户数，还有每天的付费转化率，看看哪天转化率最高。

第二轮：Agent 生成并执行 SQL

Assistant: 好的，我将从 dwd_user_active_log（用户活跃日志）、dim_user_info（用户画像表）和 dwd_pay_order（支付订单表）中查询相关数据。现在执行 SQL 查询。
{
    "tool_name": "execute_sql",
    "arguments": {
        "sql": """
        WITH daily_metrics AS (
            SELECT 
                DATE(a.active_time) AS dt

In [53]:
text_with_newlines = context

# 使用 repr() 可以最快地将不可见字符转义为可视化字符
# [1:-1] 是为了去掉 repr 默认添加的首尾单引号
print("打印结果（显示 \\n）：")
print(repr(text_with_newlines)[1:-1])

打印结果（显示 \n）：
\n下面的回答分三块：\n\n1. 先把你说的四个“流派”系统化补完，并结合近期论文/产品评价一下优劣和适用场景  \n2. 结合银行数字员工（多轮工具调用、内网 DeepSeek‑V4‑Flash）分析哪几类技术在工程上真有性价比  \n3. 给出一个「可落地的」上下文优化 / 压缩方案（含策略、数据结构、触发机制、以及如何避坑）\n\n---\n\n## 一、四类上下文 / Prompt 压缩流派系统梳理与评价\n\n### 1. 直接调大模型做提示词压缩 / SFT/DPO/RLHF 学会压缩\n\n**典型做法**\n\n- 让同一个或更强的 LLM执行：  \n  - 单轮/多轮 prompt → “请在不损失关键信息的前提下，把以上内容压缩到 N tokens 内”  \n  - 进一步在系统提示里约束结构（如 bullet points、JSON schema、专用 DSL）\n- 再往前一步，是专门训练一个「压缩专家」：\n  - 用 GPT‑4 等老师模型标注大量 `(原上下文, 任务指令) → 最佳压缩上下文`，对小模型做 SFT 或 DPO  \n  - 也有工作把「是否保留某段上下文」变成二分类，让模型学会「压缩决策策略」\n\n**优点**\n\n- 极其灵活：可以整段总结、按业务字段抽取、保留计算中间推理链等  \n- 训练/推理管线比较简单：就是「先调一次压缩 LLM，再调任务 LLM」的两阶段调用  \n- 非常适合**高度结构化的业务压缩模板**：  \n  - 例如银行对话里，永远关注：客户身份 → 业务目标 → 关键数值 → 风险点 → 工具执行结论  \n  - 可以把这些字段变成 schema，强制压缩输出成结构化结果，再拼接到下游 prompt\n\n**缺点**\n\n- 成本：每次压缩本身又是一次 LLM 调用，对你现在 10w token 的场景，如果还是用 DeepSeek‑V4‑Flash 做压缩，就会非常慢  \n- 延迟：多轮 Agent 风格对话里，如果「每一轮都全量 summarization」，会直接毁掉用户体验  \n- 信息丢失不可控：如果完全交由大模型自由“摘要”，在金融场景非常危险——数字、阈值、日期经常被「合理化」或忽略。近期有专门论文在金融分析场景上验证：LLM 摘

## Out-of-Domain

As our model is only trained on meeting transcripts data from MeetingBank, here we explore its generalization ability across various benchmarks of long-context scenarios, reasoning, and in-context learning.
Although the compressor of <i>LLMLingua-2</i> is only trained on MeetingBank data, <i>LLMLingua-2</i> is also effective on <b>out-of-domain</b> data,
with its performance <b>comparable to or even surpassing</b> the SOTA <i>task-agnostic</i> compression baselines.

Below, we showcase several compression results on <a href="https://arxiv.org/abs/2308.14508">LongBench</a> and <a href="https://arxiv.org/abs/2110.14168">GSM8K</a>, including single-document QA, multi-document QA, summarization and in-context learning tasks.

### Load LongBench Prompt

In [ ]:
dataset2prompt = {
    "narrativeqa": "You are given a story, which can be either a novel or a movie script, and a question. Answer the question asconcisely as you can, using a single phrase if possible. Do not provide any explanation.\n\nStory: {context}\n\nNow, answer the question based on the story asconcisely as you can, using a single phrase if possible. Do not provide any explanation.\n\nQuestion: {input}\n\nAnswer:",
    "gov_report": "You are given a report by a government agency. Write a one-page summary of the report.\n\nReport:\n{context}\n\nNow, write a one-page summary of the report.\n\nSummary:",
    "triviaqa": "Answer the question based on the given passage. Only give me the answer and do not output any other words. The following are some examples.\n\n{context}\n\n{input}",
}

dataset2maxlen = {
    "narrativeqa": 128,
    "gov_report": 512,
    "triviaqa": 32,
}

### Single-Doc QA

In [ ]:
task = "narrativeqa"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[3]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context,
    target_token=3000,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### Multi-Doc QA

In [ ]:
task = "triviaqa"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[0]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
context_list = context.split("\nPassage:")
context_list = ["\nPassage:" + c for c in context_list]

# 2000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context_list,
    target_token=2000,
    force_tokens=["\nPassage:", ".", "?", "\n"],
    drop_consecutive=True,
    use_context_level_filter=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### Summarization

In [ ]:
task = "gov_report"
dataset = load_dataset("THUDM/LongBench", task, split="test")
sample = dataset[0]
context = sample["context"]
reference = sample["answers"]
print(reference)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    context,
    target_token=3000,
    force_tokens=["!", ".", "?", "\n"],
    drop_consecutive=True,
)

In [ ]:
# The response from original prompt, using GPT-4-32k
import json

prompt_format = dataset2prompt[task]
max_gen = int(dataset2maxlen[task])
sample["context"] = compressed_prompt["compressed_prompt"]
prompt = prompt_format.format(**sample)

message = [
    {"role": "user", "content": prompt},
]

request_data = {
    "messages": message,
    "max_tokens": max_gen,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
}
response = openai.ChatCompletion.create(
    engine="gpt-4-32k",
    **request_data,
)
print(json.dumps(response, indent=4))

### In-Context Learning (GSM8K)

In [ ]:
!wget https://raw.githubusercontent.com/FranxYao/chain-of-thought-hub/main/gsm8k/lib_prompt/prompt_hardest.txt
prompt_complex = open("./prompt_hardest.txt").read()
gsm8k = load_dataset("gsm8k", "main")
gsm8k_test = gsm8k["test"]

In [ ]:
# select an example from GSM8K
question, answer = [gsm8k_test[2][key] for key in ["question", "answer"]]
# Ground-truth Answer
print("Question:", question)
print("Answer:", answer)

In [ ]:
# The response from original prompt
import json

instruction = "Please reference the following examples to answer the math question,\n"
prompt = instruction + prompt_complex + "\n\nQuestion: " + question

request_data = {
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
    "stop": "\n\n",
}
response = openai.Completion.create(
    engine="gpt-35-turbo-instruct",
    **request_data,
)
print(json.dumps(response, indent=4))

In [ ]:
# 3000 Compression
compressed_prompt = llm_lingua.compress_prompt(
    prompt_complex.split("\n\n"),
    target_token=150,
    force_tokens=["+", "-", "*", "×", "/", "÷", "=", "The answer is", "\n"],
    drop_consecutive=True,
    force_reserve_digit=True,
    use_context_level_filter=True,
)

In [ ]:
instruction = "Please reference the following examples to answer the math question,\n"
prompt = (
    instruction + compressed_prompt["compressed_prompt"] + "\n\nQuestion: " + question
)

request_data = {
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0,
    "top_p": 1,
    "n": 1,
    "stream": False,
    "stop": "\r\n",
}
response = openai.Completion.create(
    engine="gpt-35-turbo-instruct",
    **request_data,
)
print("Response:", response)